# B2-024-gpu-scientific-ml-capstone — Practice p21 — Solution

**Type:** integrative · **Difficulty:** core · **Concepts:** open-ended-model-evaluation

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

FAMILIES = ("shallow", "deep")
SEEDS = (0, 1, 2)
ABLATION_SEEDS = (0, 1)
STEPS_PER_RUN = 250
TOTAL_BUDGET = 2000

## Solution

**Model class.** `InverseRegressor` keeps everything that is fitted (feature statistics and weights) inside `fit`, which first hashes every training row (the seam), then computes the statistics from `train_x` only, seeds, builds the family's network, wraps Adam with the shared budget, and runs full-batch MSE steps. `predict` applies the same features and returns `(N, 3)`.

**Budget arithmetic** (one `StepBudget(2000)` for the whole report): $2$ families $\times$ $3$ seeds $\times$ $250$ steps $=1{,}500$, plus the ablation $1$ family $\times$ $2$ seeds $\times$ $250 = 500$, total exactly $2{,}000$.

**Statistics.** The p08 bootstrap functions are reused with the metric "mean position error" over examples; each call gets a fresh `torch.Generator().manual_seed(SEED)`.

In [ ]:
import math

train_x, train_y = capstone_data.train_rows("inverse")
val_x, val_y = capstone_data.val_rows("inverse")
SEAM = []


def build_net(family):
    if family == "shallow":
        return nn.Sequential(nn.Linear(8, 64), nn.ReLU(), nn.Linear(64, 3))
    if family == "deep":
        return nn.Sequential(nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(),
                             nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 3))
    raise ValueError(family)


class InverseRegressor:
    def __init__(self, family, use_log, seed, steps, budget):
        self.family, self.use_log, self.seed, self.steps, self.budget = family, use_log, seed, steps, budget

    def _features(self, m):
        return torch.log(m) if self.use_log else m

    def fit(self, train_x, train_y):
        SEAM.extend(capstone_data.split_of("inverse", row) for row in train_x)
        feats = self._features(train_x)
        self.mu, self.sd = feats.mean(dim=0), feats.std(dim=0)
        torch.manual_seed(self.seed)
        self.net = build_net(self.family)
        optimizer = self.budget.wrap(torch.optim.Adam(self.net.parameters(), lr=3e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0))
        z = (feats - self.mu) / self.sd
        self.net.train()
        for _ in range(self.steps):
            optimizer.zero_grad(set_to_none=True)
            loss = F.mse_loss(self.net(z), train_y)
            loss.backward()
            optimizer.step()
        self.net.eval()
        return self

    def predict(self, x):
        with torch.no_grad():
            return self.net((self._features(x) - self.mu) / self.sd)

    def n_params(self):
        return sum(p.numel() for p in self.net.parameters())


def _quantiles(stats, alpha):
    q = torch.tensor([alpha / 2, 1 - alpha / 2], dtype=torch.float64)
    low, high = torch.quantile(torch.tensor(stats, dtype=torch.float64), q).tolist()
    return float(low), float(high)


def bootstrap_ci(metric_fn, y_true, y_pred, n_boot, alpha, generator):
    point = float(metric_fn(y_true, y_pred))
    n = y_true.shape[0]
    idx = torch.randint(0, n, (n_boot, n), generator=generator)
    stats = [float(metric_fn(y_true[idx[b]], y_pred[idx[b]])) for b in range(n_boot)]
    return (point, *_quantiles(stats, alpha))


def paired_bootstrap_diff(metric_fn, y_true, pred_a, pred_b, n_boot, alpha, generator):
    point = float(metric_fn(y_true, pred_a)) - float(metric_fn(y_true, pred_b))
    n = y_true.shape[0]
    idx = torch.randint(0, n, (n_boot, n), generator=generator)
    stats = [float(metric_fn(y_true[idx[b]], pred_a[idx[b]])) - float(metric_fn(y_true[idx[b]], pred_b[idx[b]]))
             for b in range(n_boot)]
    return (point, *_quantiles(stats, alpha))


position_metric = lambda target, pred: capstone_data.position_error_per_example(pred, target).mean()
fresh_gen = lambda: torch.Generator().manual_seed(SEED)


def mean_sd(values):
    m = sum(values) / len(values)
    return m, math.sqrt(sum((v - m) ** 2 for v in values) / (len(values) - 1))

In [ ]:
budget = capstone_data.StepBudget(TOTAL_BUDGET)
# Budget: 2 families x 3 seeds x 250 = 1500 steps; ablation 1 x 2 seeds x 250 = 500 steps; total 2000.
MODELS, VAL_ERR, PER_EXAMPLE = {}, {}, {}
for family in FAMILIES:
    for seed in SEEDS:
        reg = InverseRegressor(family, True, seed, STEPS_PER_RUN, budget).fit(train_x, train_y)
        per = capstone_data.position_error_per_example(reg.predict(val_x), val_y)
        MODELS[family, seed], PER_EXAMPLE[family, seed], VAL_ERR[family, seed] = reg, per, per.mean().item()
ABLATION = {}
for use_log in (True, False):
    for seed in ABLATION_SEEDS:
        if use_log:
            ABLATION[use_log, seed] = VAL_ERR["deep", seed]          # already trained above
        else:
            reg = InverseRegressor("deep", False, seed, STEPS_PER_RUN, budget).fit(train_x, train_y)
            ABLATION[use_log, seed] = position_metric(val_y, reg.predict(val_x)).item()

# 1. per-family table
TABLE = {}
for family in FAMILIES:
    errs = [VAL_ERR[family, s] for s in SEEDS]
    m, sd = mean_sd(errs)
    TABLE[family] = {"errors": errs, "mean": m, "std": sd, "params": MODELS[family, 0].n_params()}
    print(f"{family:8s} per-seed {[round(e, 4) for e in errs]} mean {m:.4f} sd {sd:.4f} params {TABLE[family]['params']}")
# 2. intervals for the seed-0 models
CI = {family: bootstrap_ci(position_metric, val_y, MODELS[family, 0].predict(val_x), 1000, 0.05, fresh_gen()) for family in FAMILIES}
for family, (p, lo, hi) in CI.items():
    print(f"{family} seed 0: {p:.4f} [{lo:.4f}, {hi:.4f}]")
# 3. paired comparison
seed_diffs = [VAL_ERR["deep", s] - VAL_ERR["shallow", s] for s in SEEDS]
err_deep = torch.stack([PER_EXAMPLE["deep", s] for s in SEEDS]).mean(dim=0)
err_shallow = torch.stack([PER_EXAMPLE["shallow", s] for s in SEEDS]).mean(dim=0)
PAIRED = paired_bootstrap_diff(lambda target, err: err.mean(), torch.zeros(200), err_deep, err_shallow, 1000, 0.05, fresh_gen())
print("per-seed deep - shallow:", [round(d, 4) for d in seed_diffs], "| paired:", tuple(round(v, 4) for v in PAIRED))
# 4. ablation table
ABL_TABLE = {("log" if use_log else "raw"): [ABLATION[use_log, s] for s in ABLATION_SEEDS] for use_log in (True, False)}
for name, errs in ABL_TABLE.items():
    print(f"deep, {name} features: {[round(e, 4) for e in errs]} mean {sum(errs) / 2:.4f}")
# 5. robustness
eps = torch.randn(200, 8, generator=torch.Generator().manual_seed(SEED + 1))
shifted = val_x * (1 + 0.05 * eps)
ROBUST = {family: position_metric(val_y, MODELS[family, 0].predict(shifted)).item() for family in FAMILIES}
print("shifted-input seed-0 errors:", {k: round(v, 4) for k, v in ROBUST.items()})

# Pre-registered selection rule
lower = min(FAMILIES, key=lambda fam: TABLE[fam]["mean"])
contains_zero = PAIRED[1] <= 0.0 <= PAIRED[2]
chosen_family = "shallow" if contains_zero else lower
print("budget used:", budget.used, "| lower mean:", lower, "| paired CI contains 0:", contains_zero, "| chosen:", chosen_family)

**Locked test score (one call, after all fitting and selection).**

In [ ]:
baseline = capstone_data.baseline_score("inverse")
chosen_model = MODELS[chosen_family, 0]
assert capstone_data.test_call_count("inverse") == 0
test = capstone_data.final_test_score(chosen_model.predict, n_boot=1000, alpha=0.05, task="inverse")
assert capstone_data.test_call_count("inverse") == 1
print(f"{chosen_family} seed 0: test position error {test.score:.4f}, 95% CI [{test.ci_low:.4f}, {test.ci_high:.4f}]; baseline {baseline.score:.4f}")

**Evaluation section.**

In [ ]:
tab = TABLE
report = f'''Protocol: both families were trained on the 800 training rows only (log magnitudes standardized with training statistics, full-batch Adam, 250 steps per run, shared seeds 0-2) under one 2,000-step budget, all comparisons used the 200 validation rows, and the locked test set was scored once.
Headline: the chosen {chosen_family} family's seed-0 model has a test position error of {test.score:.4f} (95% bootstrap CI [{test.ci_low:.4f}, {test.ci_high:.4f}]), far below the committed grid-Tikhonov baseline's {baseline.score:.4f}.
On validation the shallow family averages {tab["shallow"]["mean"]:.4f} (sd {tab["shallow"]["std"]:.4f}, {tab["shallow"]["params"]} parameters) and the deep family {tab["deep"]["mean"]:.4f} (sd {tab["deep"]["std"]:.4f}, {tab["deep"]["params"]} parameters).
The per-seed differences deep minus shallow are {", ".join(f"{d:+.4f}" for d in seed_diffs)}, and the paired bootstrap interval of the seed-averaged difference is [{PAIRED[1]:+.4f}, {PAIRED[2]:+.4f}] around {PAIRED[0]:+.4f}.
{"Because that interval contains 0, the pre-registered rule picks the smaller shallow family." if contains_zero else f"Because that interval excludes 0, the pre-registered rule picks the family with the lower mean, {chosen_family}."}
The seed-0 validation intervals are shallow [{CI["shallow"][1]:.4f}, {CI["shallow"][2]:.4f}] and deep [{CI["deep"][1]:.4f}, {CI["deep"][2]:.4f}].
The ablation shows that log features matter for the deep family: raw standardized magnitudes give {sum(ABL_TABLE["raw"]) / 2:.4f} against {sum(ABL_TABLE["log"]) / 2:.4f} with logs over seeds 0 and 1.
Under extra 5% multiplicative input perturbations the seed-0 errors rise to {ROBUST["shallow"]:.4f} (shallow, {100 * (ROBUST["shallow"] / CI["shallow"][0] - 1):+.0f}%) and {ROBUST["deep"]:.4f} (deep, {100 * (ROBUST["deep"] / CI["deep"][0] - 1):+.0f}%), so both degrade gradually and the ranking is unchanged.
Limitations: 250 steps is a short budget that may favour whichever family converges faster, three seeds give only a rough spread, the ablation has two seeds, and the robustness check covers only one perturbation type.
The test interval reflects example sampling for one seed-0 model, not seed-to-seed variation.'''
print(report)
n_sentences = report.count(".\n") + 1
print("sentences:", n_sentences)

### Answer check

In [ ]:
assert budget.used == 2000
assert len(SEAM) == 8 * 800 and set(SEAM) == {"train"}
for p, lo, hi in list(CI.values()) + [PAIRED]:
    assert lo <= p <= hi
expected_choice = "shallow" if PAIRED[1] <= 0.0 <= PAIRED[2] else min(FAMILIES, key=lambda fam: TABLE[fam]["mean"])
assert chosen_family == expected_choice
assert capstone_data.test_call_count("inverse") == 1
assert test.score < baseline.score
assert len(TABLE) == 2 and all(len(TABLE[f]["errors"]) == 3 for f in FAMILIES)
assert len(ABL_TABLE) == 2 and all(len(v) == 2 for v in ABL_TABLE.values())
assert n_sentences <= 12
assert (ROBUST["deep"] < ROBUST["shallow"]) == (TABLE["deep"]["mean"] < TABLE["shallow"]["mean"])   # "ranking unchanged"